Make a "Group E" and "Group F".  Using the cleaned data. Both groups should have an initial score in Initial 2022, they should have summative tests in 2022, 2023, 2024 with dates after the initial test. They should have enrollment data for all three years. Group E should have an initial assessment of Novice Learner, and Group F should have Intermediate Learner

In [4]:
import pandas as pd
import os

# --- Paths ---
elpac_clean_dir = "../1095 Data/ELPAC/Cleaned"
enrollment_dir  = "../1095 Data/Enrollment"

# --- Load Cleaned Initial 2022 only ---
initial_22 = pd.read_csv(os.path.join(elpac_clean_dir, "InitialElpac_22_clean.csv"))
initial_22 = initial_22.apply(lambda col: col.str.strip() if col.dtype == "object" else col)
initial_22["TestStartDate"] = pd.to_datetime(initial_22["TestStartDate"], errors="coerce")
initial_22["OverallScaleScore"] = pd.to_numeric(initial_22["OverallScaleScore"], errors="coerce")
print(f"Initial 2022 (clean): {len(initial_22):,} rows")

# --- Load Cleaned Summative files ---
summative_files = {
    2022: "SummativeElpac_22_clean.csv",
    2023: "SummativeElpac_23_clean.csv",
    2024: "SummativeElpac_24_clean.csv",
}

summative_dfs = {}
for year, filename in summative_files.items():
    df = pd.read_csv(os.path.join(elpac_clean_dir, filename))
    df = df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)
    df["OverallScaleScore"] = pd.to_numeric(df["OverallScaleScore"], errors="coerce")
    df["TestStartDate"] = pd.to_datetime(df["TestStartDate"], errors="coerce")
    summative_dfs[year] = df
    print(f"Summative {year} (clean): {len(df):,} rows")

# --- Load Enrollment files ---
enrollment_files = {
    "2021-22": "CEnrollment_21-22.csv",
    "2022-23": "CEnrollment_22-23.csv",
    "2023-24": "CEnrollment_23-24.csv",
}

enrollment_dfs = {}
for label, filename in enrollment_files.items():
    enrollment_dfs[label] = pd.read_csv(os.path.join(enrollment_dir, filename), low_memory=False)
    print(f"Enrollment {label}: {len(enrollment_dfs[label]):,} rows")

# --- SSIDs present in all 3 enrollment years ---
ssids_enroll_2122 = set(enrollment_dfs["2021-22"]["SSID"].dropna())
ssids_enroll_2223 = set(enrollment_dfs["2022-23"]["SSID"].dropna())
ssids_enroll_2324 = set(enrollment_dfs["2023-24"]["SSID"].dropna())
ssids_all_enrollment = ssids_enroll_2122 & ssids_enroll_2223 & ssids_enroll_2324
print(f"\nSSIDs with enrollment in all 3 years: {len(ssids_all_enrollment):,}")

# --- Build lookup: SSID -> initial date from Initial 2022 ---
initial_dates = (
    initial_22[["SSID", "TestStartDate", "OverallPL"]]
    .dropna(subset=["TestStartDate", "OverallPL"])
    .set_index("SSID")
)

# --- Build lookup: SSID -> summative date per year ---
summative_dates = {}
for year, df in summative_dfs.items():
    valid = df[["SSID", "TestStartDate"]].dropna()
    summative_dates[year] = dict(zip(valid["SSID"], valid["TestStartDate"]))

# --- Filter function ---
def build_group(target_pl, min_score=1150):
    group_ssids = []
    candidates = initial_dates[initial_dates["OverallPL"] == target_pl].index

    for ssid in candidates:
        # Must have enrollment in all 3 years
        if ssid not in ssids_all_enrollment:
            continue

        init_date = initial_dates.loc[ssid, "TestStartDate"]
        if pd.isna(init_date):
            continue

        # Must have summative records in all 3 years, each after initial date
        # AND each score must be >= min_score
        has_all_summatives = all(
            ssid in summative_dates[year]
            and summative_dates[year][ssid] > init_date
            and summative_scores[year].get(ssid, 0) >= min_score
            for year in [2022, 2023, 2024]
        )

        if has_all_summatives:
            group_ssids.append(ssid)

    return group_ssids
# --- Build Groups E and F ---
# group_e_ssids = build_group("Novice English Learner")
# group_f_ssids = build_group("Intermediate English Learner")

# print(f"\nGroup E (Novice, Initial 2022, 3 summatives after initial, 3 enrollment years):        {len(group_e_ssids):,} students")
# print(f"Group F (Intermediate, Initial 2022, 3 summatives after initial, 3 enrollment years): {len(group_f_ssids):,} students")

# --- Export to CSV ---
# pd.DataFrame({"SSID": group_e_ssids}).to_csv("group_e_ssids.csv", index=False)
# pd.DataFrame({"SSID": group_f_ssids}).to_csv("group_f_ssids.csv", index=False)
# print("\nSaved: group_e_ssids.csv")
# print("Saved: group_f_ssids.csv")

Initial 2022 (clean): 62,946 rows


/var/folders/63/0syx5ts177gcl1c62wf8lbywrc37vm/T/ipykernel_31452/591691076.py:24: DtypeWarning: Columns (0: OverallPL, 1: OralLanguagePL, 2: WrittenLanguagePL) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(elpac_clean_dir, filename))


Summative 2022 (clean): 302,476 rows


/var/folders/63/0syx5ts177gcl1c62wf8lbywrc37vm/T/ipykernel_31452/591691076.py:24: DtypeWarning: Columns (0: OverallPL, 1: OralLanguagePL, 2: WrittenLanguagePL) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(elpac_clean_dir, filename))


Summative 2023 (clean): 296,681 rows


/var/folders/63/0syx5ts177gcl1c62wf8lbywrc37vm/T/ipykernel_31452/591691076.py:24: DtypeWarning: Columns (0: OverallPL, 1: OralLanguagePL, 2: WrittenLanguagePL) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(elpac_clean_dir, filename))


Summative 2024 (clean): 291,920 rows



KeyboardInterrupt



In [2]:
# --- Fix 1: suppress DtypeWarning by adding low_memory=False to summative loading ---
# Update the summative loading loop:
for year, filename in summative_files.items():
    df = pd.read_csv(os.path.join(elpac_clean_dir, filename), low_memory=False)
    df = df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)
    df["OverallScaleScore"] = pd.to_numeric(df["OverallScaleScore"], errors="coerce")
    df["TestStartDate"] = pd.to_datetime(df["TestStartDate"], errors="coerce")
    summative_dfs[year] = df

# --- Fix 2: Debug the zero students issue ---

# Step 1: Check OverallPL values
print("=== Unique OverallPL values in Initial 2022 ===")
for val, count in initial_22["OverallPL"].value_counts().items():
    print(f"  '{val}': {count:,}")

# Step 2: Check how many initial 2022 SSIDs are in all 3 enrollment years
initial_ssids = set(initial_22["SSID"].dropna())
in_enrollment = initial_ssids & ssids_all_enrollment
print(f"\n=== Initial 2022 SSIDs also in all 3 enrollment years: {len(in_enrollment):,} ===")

# Step 3: Check how many of those are in all 3 summative files
in_all_summatives = in_enrollment
for year, df in summative_dfs.items():
    in_all_summatives = in_all_summatives & set(df["SSID"].dropna())
print(f"=== Of those, also in all 3 summative files: {len(in_all_summatives):,} ===")

# Step 4: Check date comparison for a sample student
sample_ssid = next(iter(in_all_summatives))
print(f"\n=== Sample SSID: {sample_ssid} ===")
sample_initial = initial_22[initial_22["SSID"] == sample_ssid][["SSID", "TestStartDate", "OverallPL"]]
print(f"Initial record:\n{sample_initial}")
for year, df in summative_dfs.items():
    sample_summ = df[df["SSID"] == sample_ssid][["SSID", "TestStartDate", "OverallScaleScore"]]
    print(f"Summative {year}:\n{sample_summ}")

=== Unique OverallPL values in Initial 2022 ===
  'Novice English Learner                  ': 45,541
  'Intermediate English Learner            ': 8,799
  'Initial Fluent English Proficient (IFEP)': 7,262
  '                                        ': 936
  'NS                                      ': 408

=== Initial 2022 SSIDs also in all 3 enrollment years: 50,213 ===
=== Of those, also in all 3 summative files: 35,205 ===

=== Sample SSID: 4957929472 ===
Initial record:
             SSID TestStartDate                                 OverallPL
48032  4957929472    2021-11-24  Novice English Learner                  
Summative 2022:
              SSID TestStartDate  OverallScaleScore
255886  4957929472    2022-03-09             1395.0
Summative 2023:
              SSID TestStartDate  OverallScaleScore
166047  4957929472    2023-02-16             1473.0
Summative 2024:
              SSID TestStartDate  OverallScaleScore
152346  4957929472    2024-04-29             1474.0


In [3]:
# --- Explicitly strip OverallPL and rebuild initial_dates ---
initial_22["OverallPL"] = initial_22["OverallPL"].astype(str).str.strip()

# Verify the fix
print("=== OverallPL values after strip ===")
for val, count in initial_22["OverallPL"].value_counts().items():
    print(f"  '{val}': {count:,}")

# --- Rebuild initial_dates lookup ---
initial_dates = (
    initial_22[["SSID", "TestStartDate", "OverallPL"]]
    .dropna(subset=["TestStartDate", "OverallPL"])
    .set_index("SSID")
)

# --- Rebuild Groups E and F ---
# group_e_ssids = build_group("Novice English Learner")
# group_f_ssids = build_group("Intermediate English Learner")

# print(f"\nGroup E (Novice):       {len(group_e_ssids):,} students")
# print(f"Group F (Intermediate): {len(group_f_ssids):,} students")

# # --- Export to CSV ---
# pd.DataFrame({"SSID": group_e_ssids}).to_csv("group_e_ssids.csv", index=False)
# pd.DataFrame({"SSID": group_f_ssids}).to_csv("group_f_ssids.csv", index=False)
# print("\nSaved: group_e_ssids.csv")
# print("Saved: group_f_ssids.csv")

=== OverallPL values after strip ===
  'Novice English Learner': 45,541
  'Intermediate English Learner': 8,799
  'Initial Fluent English Proficient (IFEP)': 7,262
  '': 936
  'NS': 408

Group E (Novice):       29,125 students
Group F (Intermediate): 4,202 students

Saved: group_e_ssids.csv
Saved: group_f_ssids.csv


In [5]:
# --- Rebuild summative lookups: dates AND scores ---
summative_dates  = {}
summative_scores = {}

for year, df in summative_dfs.items():
    valid = df[["SSID", "TestStartDate", "OverallScaleScore"]].dropna()
    summative_dates[year]  = dict(zip(valid["SSID"], valid["TestStartDate"]))
    summative_scores[year] = dict(zip(valid["SSID"], valid["OverallScaleScore"]))

# --- Rebuild Groups E and F ---
group_e_ssids = build_group("Novice English Learner")
group_f_ssids = build_group("Intermediate English Learner")

print(f"\nGroup E (Novice):       {len(group_e_ssids):,} students")
print(f"Group F (Intermediate): {len(group_f_ssids):,} students")

# --- Export to CSV ---
pd.DataFrame({"SSID": group_e_ssids}).to_csv("group_e_ssids.csv", index=False)
pd.DataFrame({"SSID": group_f_ssids}).to_csv("group_f_ssids.csv", index=False)
print("\nSaved: group_e_ssids.csv")
print("Saved: group_f_ssids.csv")


Group E (Novice):       29,074 students
Group F (Intermediate): 4,198 students

Saved: group_e_ssids.csv
Saved: group_f_ssids.csv
